# Track B - CPU prep and validation (no GPU)

Runs everything that does not need an accelerator, so the GPU run is shorter and de-risked:

1. clone the repo at the pinned commit
2. build the Python 3.11 venv with the pinned OpenMMLab stack
3. fetch the epoch-16 checkpoint and re-apply the two mmrotate source patches
4. download official DOTA v2.0, tile 1024/200, and **verify it is the right dataset**
5. extract the epoch-16 config from the checkpoint, build the model on CPU, build both
   dataloaders and pull a real sample from each

Step 4 is the one that matters: it proves the labels feeding the tiler are the official
18-class DOTA v2.0 annotations with difficulty flags, not the 16-class Ultralytics conversion.

In [ ]:
#@title 1 - Environment: GPU, repo, disk, paths { display-mode: "form" }
import subprocess, sys, os
from pathlib import Path

BASE = Path('/tmp/ada')
REPO = BASE / 'ADA'
DATA = Path('/tmp/data')
VENV = Path('/tmp/mmrotate_venv')
WORK = Path('/kaggle/working/work_dirs/trackB_official_ep17-24')
OUT = Path('/kaggle/working/ada_trackB_official')
COMMIT = 'a2ecd69d3d487c3f639c3cec329e0a737454370e'

# Kaggle exports MPLBACKEND=module://matplotlib_inline.backend_inline; that value is
# invalid outside IPython and breaks any child script that touches matplotlib.
os.environ['MPLBACKEND'] = 'Agg'

try:
    print(subprocess.run(['nvidia-smi', '--query-gpu=index,name,memory.total', '--format=csv,noheader'],
                         capture_output=True, text=True).stdout.strip() or 'no GPU on this runtime')
except FileNotFoundError:
    print('no GPU on this runtime (CPU kernel)')
print('system python', sys.version.split()[0])
print(subprocess.run(['df', '-BG', '/tmp', '/kaggle/working'], capture_output=True, text=True).stdout)

for p in (BASE, DATA, WORK, OUT, OUT / 'logs'):
    p.mkdir(parents=True, exist_ok=True)

# the config inside epoch_16.pth hard-codes /content/data/split_ss_dotav2
Path('/content').mkdir(parents=True, exist_ok=True)
subprocess.run(['rm', '-rf', '/content/data'])
Path('/content/data').symlink_to(DATA)
print('/content/data ->', Path('/content/data').resolve())

if not (REPO / 'scripts').is_dir():
    subprocess.run(['git', 'clone', '-q',
                    'https://github.com/Lion4467cat/dotav2-obb-benchmark.git', str(REPO)], check=True)
subprocess.run(['git', '-C', str(REPO), 'fetch', '-q', 'origin'], check=True)
subprocess.run(['git', '-C', str(REPO), 'checkout', '-q', COMMIT], check=True)
print('repo at', subprocess.run(['git', '-C', str(REPO), 'rev-parse', '--short', 'HEAD'],
                                capture_output=True, text=True).stdout.strip())
try:
    n_gpu = len(subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).stdout.splitlines())
except FileNotFoundError:
    n_gpu = 0
print('GPUs visible:', n_gpu)
print('(CPU kernel: GPU assertion intentionally skipped)')

In [ ]:
#@title 2 - Provision the Python 3.11 venv with the pinned OpenMMLab stack { display-mode: "form" }
import subprocess, time, os
from pathlib import Path

# Kaggle exports MPLBACKEND=module://matplotlib_inline.backend_inline, which is not a
# valid backend outside IPython and makes scripts/verify_env.py fail. Force Agg.
CHILD_ENV = dict(os.environ)
CHILD_ENV['MPLBACKEND'] = 'Agg'

if (VENV / 'bin' / 'python').is_file():
    print('venv already present at', VENV)
else:
    t0 = time.time()
    p = subprocess.Popen(['bash', str(REPO / 'scripts' / 'setup_colab.sh'), str(REPO), str(VENV)],
                         env=CHILD_ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                         text=True, bufsize=1)
    for line in p.stdout:
        print(line, end='', flush=True)
    p.wait()
    print()
    print('setup exit', p.returncode, 'after', round((time.time() - t0) / 60, 1), 'min')
    if p.returncode != 0:
        # on a CPU box the only failing step is verify_env.py's CUDA check; the venv itself is fine
        probe = subprocess.run(
            [str(VENV / 'bin' / 'python'), '-c',
             'import torch, mmcv, mmdet, mmengine; print("imports ok", torch.__version__, mmcv.__version__)'],
            capture_output=True, text=True)
        print(probe.stdout.strip())
        assert probe.returncode == 0, 'the venv is genuinely broken: ' + probe.stderr[-800:]
        print('setup returned non-zero, but the venv imports cleanly - continuing without a GPU')
print('venv python', (VENV / 'bin' / 'python').resolve())

In [ ]:
#@title 3 - Verify the venv, stage epoch_16.pth, apply the two mmrotate patches { display-mode: "form" }
import subprocess
from pathlib import Path

PY = VENV / 'bin' / 'python'

PROBE = '''
import torch, mmcv, mmengine, mmdet, os, mmrotate
print('torch', torch.__version__, '| cuda', torch.cuda.is_available(), '|', torch.version.cuda)
print('device count', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print('  gpu', i, torch.cuda.get_device_name(i),
          round(torch.cuda.get_device_properties(i).total_memory / 1024**3, 1), 'GiB')
print('mmcv', mmcv.__version__, '| mmengine', mmengine.__version__, '| mmdet', mmdet.__version__)
print('mmrotate', os.path.dirname(mmrotate.__file__))
'''
r = subprocess.run([str(PY), '-c', PROBE], capture_output=True, text=True)
print(r.stdout)
assert r.returncode == 0, r.stderr[-2000:]

CKPT16 = Path('/tmp/epoch_16.pth')
if not CKPT16.is_file() or CKPT16.stat().st_size < 700e6:
    print('downloading epoch_16.pth from Drive by file id (gdown)')
    subprocess.run([str(VENV / 'bin' / 'gdown'),
                    '12I5f7mNGUGJpMCIVKBLh1IekMhDkrpPs', '-O', str(CKPT16)], check=True)
print('checkpoint', CKPT16, round(CKPT16.stat().st_size / 1e6), 'MB')
assert CKPT16.stat().st_size > 700e6, 'checkpoint download looks truncated'

# same two source patches the original run used: analytical 2x2 determinant in the
# Gaussian KLD loss (avoids a cuSOLVER error on T4) and chunked rotated-IoU overlaps
PATCH = '''
import sys
from pathlib import Path
sys.path.insert(0, 'REPO')
from scripts.run_kaggle import patch_gaussian_dist_loss, patch_rotate_iou2d_calculator
root = Path('REPO')
patch_gaussian_dist_loss(root)
patch_rotate_iou2d_calculator(root)
'''.replace('REPO', str(REPO))
r = subprocess.run([str(PY), '-c', PATCH], capture_output=True, text=True)
print(r.stdout.strip())
assert r.returncode == 0, r.stderr[-2000:]

In [ ]:
#@title 4 - Official DOTA v2.0 -> 1024/200 tiling, then PROVE it is the right dataset { display-mode: "form" }
import subprocess, time
from collections import Counter
from pathlib import Path

SPLIT = DATA / 'split_ss_dotav2'
RAW = DATA / 'dotav2_raw'
CACHE = DATA / 'raw_archives'
PY = VENV / 'bin' / 'python'

def sh(cmd, cwd=None):
    print('+', ' '.join(str(c) for c in cmd), flush=True)
    p = subprocess.run([str(c) for c in cmd], cwd=str(cwd) if cwd else None)
    assert p.returncode == 0, 'command failed: ' + ' '.join(str(c) for c in cmd)

t0 = time.time()
have_patches = (SPLIT / 'train' / 'annfiles').is_dir() and any((SPLIT / 'train' / 'images').glob('*.png'))
import shutil as _sh, os

def _free(paths):
    # Kaggle caps the writable layer: drop every intermediate we no longer need
    # BEFORE the tiler writes ~30 GB of patches, or /tmp flips read-only (EROFS).
    for p in paths:
        try:
            if p.is_dir():
                _sh.rmtree(p, ignore_errors=True)
            elif p.is_file():
                p.unlink()
        except OSError:
            pass

if not have_patches:
    sh([PY, REPO / 'scripts' / 'download_dotav2.py', '--source', 'huggingface',
        '--cache-dir', CACHE, '--out-dir', RAW], cwd=REPO)
    # download_dotav2.py has staged RAW/<split>/{images,labelTxt}; its _extracted
    # tree and raw .tar.gz archives are now redundant - free them immediately.
    _free([RAW / '_extracted', *list(CACHE.rglob('*.tar.gz')), *list(CACHE.rglob('*.zip'))])

    # the v2.0 image tarballs hold ONLY the images new in v2.0 (419 train / 135 val).
    # the full splits are v1.0 + v2.0 = 1830 train + 593 val, so pull the v1.0 images
    # and merge them in before tiling. the v2.0 annotations already cover all of them.
    for split, url in (
        ('train', 'https://huggingface.co/datasets/isaaccorley/dota/resolve/main/dotav1.0_images_train.tar.gz'),
        ('val',   'https://huggingface.co/datasets/isaaccorley/dota/resolve/main/dotav1.0_images_val.tar.gz'),
    ):
        arc = CACHE / split / url.rsplit('/', 1)[-1]
        if not arc.is_file():
            sh(['curl', '-sL', '--retry', '3', '-o', str(arc), url])
        tmp = CACHE / ('_v1_' + split)
        _sh.rmtree(tmp, ignore_errors=True)
        tmp.mkdir(parents=True)
        sh(['tar', 'xzf', str(arc), '-C', str(tmp)])
        imgdirs = [p for p in tmp.rglob('images') if p.is_dir()]
        assert imgdirs, 'no images dir found in ' + url
        best = max(imgdirs, key=lambda p: len(list(p.glob('*.png'))))
        (RAW / split / 'images').mkdir(parents=True, exist_ok=True)
        n = 0
        for f in best.glob('*.png'):
            dst = RAW / split / 'images' / f.name
            if not dst.exists():
                # same filesystem under /tmp/data -> hardlink, not a 2nd full copy
                try:
                    os.link(f, dst)
                except OSError:
                    _sh.copy2(f, dst)
                n += 1
        print('v1.0', split, 'images merged:', n, '->', RAW / split / 'images')
        # drop this v1.0 archive + its extracted staging copy before the next split
        _free([tmp, arc])

    # Tile one split at a time using the SAME functions/config as the standalone
    # prepare_dotav2_patches.py, then delete that split's raw imagery once its
    # patches are on disk. This keeps the raw+patch disk peak far below quota.
    _driver = Path('/tmp/tile_one_split.py')
    _driver.write_text(
        'import sys, importlib.util\n'
        'from pathlib import Path\n'
        'spec = importlib.util.spec_from_file_location("_pd", sys.argv[1])\n'
        'pd = importlib.util.module_from_spec(spec); spec.loader.exec_module(pd)\n'
        'raw, splitdir, mmrotate, split = Path(sys.argv[2]), Path(sys.argv[3]), Path(sys.argv[4]), sys.argv[5]\n'
        'src = raw / split; save = splitdir / split\n'
        'import shutil as _sh\n'
        'if save.exists(): _sh.rmtree(save)\n'
        'cfg = pd.build_split_config([str(src / "images")], [str(src / "labelTxt")], save, 1024, 200, 4)\n'
        'pd.run_img_split(mmrotate, cfg)\n'
        'print("[tile]", split, "patches", len(list((save / "images").glob("*.png"))))\n'
    )
    for split in ('train', 'val'):
        sh([PY, _driver, REPO / 'scripts' / 'prepare_dotav2_patches.py', RAW, SPLIT,
            REPO / 'mmrotate', split], cwd=REPO)
        # keep labelTxt (small SOURCE annotations needed by the audit below);
        # the heavy raw imagery is no longer needed once its patches exist.
        _free([RAW / split / 'images'])
print('data ready in', round((time.time() - t0) / 60, 1), 'min')

def scan(ann_dir):
    classes, flags, inst = Counter(), Counter(), 0
    files = list(Path(ann_dir).glob('*.txt'))
    for f in files:
        for line in f.read_text(errors='ignore').splitlines():
            p = line.split()
            if len(p) < 10 or p[0].lower() in ('imagesource:', 'gsd:'):
                continue
            classes[p[8]] += 1
            flags[p[9]] += 1
            inst += 1
    return len(files), inst, classes, flags

V2_CLASSES = {'plane', 'ship', 'storage-tank', 'baseball-diamond', 'tennis-court',
              'basketball-court', 'ground-track-field', 'harbor', 'bridge', 'large-vehicle',
              'small-vehicle', 'helicopter', 'roundabout', 'soccer-ball-field',
              'swimming-pool', 'container-crane', 'airport', 'helipad'}

def ann_root(kind):
    # the tiler consumes RAW/<split>/labelTxt, so that is what must be proven official
    direct = RAW / kind / 'labelTxt'
    if direct.is_dir():
        return direct
    hits = [p for p in RAW.rglob('version2.0') if p.is_dir() and kind in str(p)]
    return hits[0] if hits else None

src_train, src_val = ann_root('train'), ann_root('val')
print('labels that feed the tiler:', src_train, '|', src_val)
tr_f, tr_i, tr_c, tr_fg = scan(src_train)
va_f, va_i, va_c, va_fg = scan(src_val)
print('SOURCE train files', tr_f, 'instances', tr_i, 'classes', len(tr_c))
print('SOURCE val   files', va_f, 'instances', va_i, 'classes', len(va_c))
print('SOURCE train difficulty flags', dict(tr_fg))
print('SOURCE classes', sorted(tr_c))

p_tr_f, p_tr_i, p_tr_c, _ = scan(SPLIT / 'train' / 'annfiles')
p_va_f, p_va_i, p_va_c, p_va_fg = scan(SPLIT / 'val' / 'annfiles')
n_tr_img = len(list((SPLIT / 'train' / 'images').glob('*.png')))
n_va_img = len(list((SPLIT / 'val' / 'images').glob('*.png')))
print()
print('PATCH train', n_tr_img, 'images /', p_tr_f, 'annfiles /', p_tr_i, 'instances')
print('PATCH val  ', n_va_img, 'images /', p_va_f, 'annfiles /', p_va_i, 'instances')
print('PATCH val non-difficult', p_va_fg.get('0', 0), '| difficult(diff!=0)', sum(v for k, v in p_va_fg.items() if k != '0'), '(flag1', p_va_fg.get('1', 0), 'flag2', p_va_fg.get('2', 0), ')')

checks = []
def check(name, ok, detail=''):
    checks.append((name, bool(ok)))
    print(('  PASS  ' if ok else '  FAIL  ') + name + ('   ' + str(detail) if detail else ''))

print()
check('source train annfiles == 1830', tr_f == 1830, tr_f)
check('source train instances == 268627', tr_i == 268627, tr_i)
check('source val annfiles >= 500', va_f >= 500, va_f)
check('class set is exactly DOTA v2.0 (18 classes)', set(tr_c) == V2_CLASSES,
      sorted(V2_CLASSES ^ set(tr_c)) or 'exact match')
check('airport present', 'airport' in tr_c, tr_c.get('airport', 0))
check('helipad present', 'helipad' in tr_c, tr_c.get('helipad', 0))
check('container-crane present', 'container-crane' in tr_c, tr_c.get('container-crane', 0))
check('difficulty flags present at source level',
      tr_fg.get('0', 0) > 0 and tr_fg.get('1', 0) > 0,
      'flag0=' + str(tr_fg.get('0', 0)) + ' flag1=' + str(tr_fg.get('1', 0)))
check('val patches == 7122', n_va_img == 7122, n_va_img)
check('val non-difficult instances == 69542', p_va_fg.get('0', 0) == 69542, p_va_fg.get('0', 0))
check('val difficult instances == 70561 (diff!=0)', sum(v for k, v in p_va_fg.items() if k != '0') == 70561, 'flag1=' + str(p_va_fg.get('1', 0)) + ' flag2=' + str(p_va_fg.get('2', 0)))
check('train patches == 24151 (15749 v1.0 + 8402 v2.0-new)', n_tr_img == 24151, n_tr_img)
check('18 classes at patch level', len(p_tr_c) == 18, len(p_tr_c))

tr_names = {p.stem for p in (SPLIT / 'train' / 'images').glob('*.png')}
va_names = {p.stem for p in (SPLIT / 'val' / 'images').glob('*.png')}
check('no train/val patch overlap', len(tr_names & va_names) == 0, len(tr_names & va_names))

failed = [n for n, ok in checks if not ok]
assert not failed, 'DATASET VERIFICATION FAILED: ' + ', '.join(failed)
print()
print('dataset verification: all', len(checks), 'checks passed')

In [ ]:
#@title 5 - Validate config, model and dataloaders on CPU { display-mode: "form" }
import subprocess
from pathlib import Path

PY = VENV / 'bin' / 'python'
PREP_PATH = Path('/tmp/prep_check.py')
PREP_PATH.write_text('import json, os, sys\nfrom pathlib import Path\nimport torch\nfrom mmengine.config import Config\nfrom mmengine.registry import MODELS, DATASETS, init_default_scope\nimport mmrotate  # noqa: F401\n\nWORK = Path(\'/kaggle/working/work_dirs/trackB_official_ep17-24\')\nWORK.mkdir(parents=True, exist_ok=True)\nCFG16 = WORK / \'ep16_cfg.py\'\n\nck = torch.load(\'/tmp/epoch_16.pth\', map_location=\'cpu\')\nCFG16.write_text(ck[\'meta\'][\'cfg\'])\ndel ck\nprint(\'[cfg] extracted from epoch_16.pth:\', len(CFG16.read_text()), \'chars\')\n\nbase = CFG16.read_text()\nextra = """\nwork_dir = \'%s\'\nresume = True\nload_from = \'/tmp/epoch_16.pth\'\ntrain_dataloader[\'batch_size\'] = 1\noptim_wrapper[\'accumulative_counts\'] = 4\ntrain_cfg[\'val_interval\'] = 2\n""" % WORK\nTRAIN_CFG = WORK / \'train_cfg.py\'\nTRAIN_CFG.write_text(base + extra)\nprint(\'[cfg] training config written:\', TRAIN_CFG)\n\ninit_default_scope(\'mmrotate\')\ncfg = Config.fromfile(str(TRAIN_CFG))\nprint(\'[cfg] max_epochs\', cfg.train_cfg[\'max_epochs\'], \'| val_interval\', cfg.train_cfg[\'val_interval\'])\nprint(\'[cfg] base_lr\', cfg.optim_wrapper[\'optimizer\'][\'lr\'], \'| accumulative_counts\', cfg.optim_wrapper[\'accumulative_counts\'])\nprint(\'[cfg] param_schedulers\', [s[\'type\'] for s in cfg.param_scheduler])\nprint(\'[cfg] data_root\', cfg.data_root)\n\nmodel = MODELS.build(cfg.model)\nn = sum(p.numel() for p in model.parameters())\nprint(\'[model] built on CPU | params %.2f M\' % (n / 1e6))\nhead = getattr(model, \'bbox_head\', None)\nprint(\'[model] bbox_head classes\', getattr(head, \'num_classes\', \'n/a\'))\n\nds = DATASETS.build(cfg.train_dataloader[\'dataset\'])\nprint(\'[data] train dataset len\', len(ds))\nprint(\'[data] classes\', len(ds.metainfo[\'classes\']), ds.metainfo[\'classes\'])\nsample = ds[0]\nprint(\'[data] sample keys\', sorted(sample.keys()))\nprint(\'[data] img shape\', tuple(sample[\'inputs\'].shape))\ngts = sample[\'data_samples\'].gt_instances\nprint(\'[data] gt count\', len(gts.labels), \'| labels\', sorted(set(gts.labels.tolist())))\nprint(\'[data] bbox shape\', tuple(gts.bboxes.shape))\n\nval_ds = DATASETS.build(cfg.val_dataloader[\'dataset\'])\nprint(\'[data] val dataset len\', len(val_ds))\nvs = val_ds[0]\nprint(\'[data] val img shape\', tuple(vs[\'inputs\'].shape))\nprint(\'[data] val metainfo classes\', len(val_ds.metainfo[\'classes\']))\n\nprint()\nprint(\'ALL PREP CHECKS PASSED - config, model and both dataloaders are sound\')\n')

env = dict(os.environ)
env['MPLBACKEND'] = 'Agg'
env['PYTHONPATH'] = str(REPO) + ':' + str(REPO / 'mmrotate') + ':' + env.get('PYTHONPATH', '')

print('+', PY, PREP_PATH)
p = subprocess.run([str(PY), str(PREP_PATH)], cwd=str(REPO), env=env, capture_output=True, text=True)
print(p.stdout)
if p.returncode:
    print(p.stderr[-3000:])
assert p.returncode == 0, 'prep validation failed'